# Learned power control

> Channel inversion with a learned prediction of the gain: the predictor, its margin and scheduling threshold chosen for a target reliability, and C3-WM's link as a pipeline of comm-core blocks with the power control as one of them.

In [ ]:
#| default_exp comm.power_control

In [ ]:
#| hide
from nbdev.showdoc import *

The problem, per directed link and step: the least power that reaches the target SNR `Γ` with probability at least
`1 − ε`, within the budget `ρ_max`:

```text
minimize ρ   subject to   P(SNR(ρ) ≥ Γ) ≥ 1 − ε,   0 ≤ ρ ≤ ρ_max
```

The transmitter predicts the gain (`ĝ`, the lowest prediction over the receiver's possible cells), and

* sends at `ρ̂ = 10^(κ/10) Γσ²/ĝ`, capped at `ρ_max`, with a margin `κ` (dB) against the prediction's errors;
* withholds the transmission if `ρ̂ > 10^(τ/10) ρ_max`: `τ = 0` withholds as soon as the budget is exceeded
  (C3-WM's rule), `τ = ∞` never does (best effort).

`κ` and `τ` are chosen on calibration layouts (held out from training): the pair with the least power that reaches
the target there. A transmission counts if it is *servable* (perfect CSI could deliver it: `g ≥ g_min = Γσ²/ρ_max`);
reliability is the share of the servable transmissions reaching `Γ`, and power is relative to perfect CSI's (exactly
`Γσ²/g` for each servable transmission, nothing for the others). See comm-core's example *Power control on new
layouts* for the study behind these choices.

In [ ]:
#| export
from __future__ import annotations

import json
import math
import pickle
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np

from c3jepa_wm.comm.channel_data import DIRS, db, features, step_features

## The link budget

In [ ]:
#| export
@dataclass
class Budget:
    "The link budget: what `κ`, `τ` and the reliability are judged against."
    snr_target_db: float = 10.0 # Γ
    noise_dbm: float = -94.0 # σ²
    max_power_dbm: float = 20.0 # ρ_max

    @property
    def g_min(self) -> float:
        "The weakest gain that can be served: Γσ²/ρ_max."
        return 10 ** ((self.snr_target_db + self.noise_dbm - self.max_power_dbm) / 10)

    @property
    def noise_w(self) -> float: return 10 ** ((self.noise_dbm - 30) / 10)

    @property
    def max_power_w(self) -> float: return 10 ** ((self.max_power_dbm - 30) / 10)

## The predictor

In [ ]:
#| export
def fit_predictor(
    X: np.ndarray, # Features to the receiver's actual cell (`build_dataset`'s `X`)
    gain: np.ndarray, # The gains |h|²
    quantile: float | None = 0.05, # The quantile of the gain (dB) predicted; None: the mean
    max_iter: int = 400, # Boosting iterations
    learning_rate: float = 0.05, # Boosting learning rate
    **kwargs, # Other options of scikit-learn's HistGradientBoostingRegressor
):
    "Gradient-boosted trees predicting the gain in dB (a quantile of it, or its mean). Early stopping is off: on, it stops quantile fits far too early."
    from sklearn.ensemble import HistGradientBoostingRegressor
    loss = dict(loss='quantile', quantile=quantile) if quantile is not None else dict(loss='squared_error')
    model = HistGradientBoostingRegressor(max_iter=max_iter, learning_rate=learning_rate, early_stopping=False,
                                          random_state=0, **loss, **kwargs)
    return model.fit(X, db(gain))

def _predict(model, X: np.ndarray, S: np.ndarray | None = None) -> np.ndarray:
    "The model's predictions (dB) from the features `X`; a per-step model (`steps`) also gets each row's past steps `S`."
    return model.predict(X, S) if getattr(model, 'steps', False) else model.predict(X)

def predict_gain(model, X_possible: np.ndarray, possible_mask: np.ndarray, S: np.ndarray | None = None) -> np.ndarray:
    """ĝ: the model's lowest prediction over the receiver's possible cells (|h|²). A per-step model also gets the
    past steps `S` (`build_dataset(..., steps=True)`'s), the same for each possible cell."""
    n, k, F = X_possible.shape
    S = None if S is None else np.repeat(S, k, axis=0)
    v = _predict(model, X_possible.reshape(n * k, F), S).reshape(n, k)
    return 10 ** (np.where(possible_mask, v, np.inf).min(1) / 10)

def predict_on(model, data: dict) -> np.ndarray:
    "`predict_gain` on a dataset (`build_dataset`'s): its possible cells, and its past steps if it has them."
    return predict_gain(model, data['X_possible'], data['possible_mask'], data.get('S'))

## Margin and threshold

In [ ]:
#| export
MARGINS_DB = np.arange(-10, 40.01, 0.25)
TAUS_DB = (0.0, 1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25, 30, math.inf)

def outcome(estimate, gain, g_min: float, tau_db: float = math.inf, margins_db=MARGINS_DB) -> dict:
    """For each margin: the share of the servable transmissions reaching Γ (`reliability`), the mean power relative to
    perfect CSI's (`power`), and the share withheld (`withheld`)."""
    margins_db = np.atleast_1d(np.asarray(margins_db, float))
    servable, need = gain >= g_min, 1 / gain                      # powers in units of Γσ²
    rho = 10 ** (margins_db[:, None] / 10) / np.maximum(estimate, 1e-30)[None, :]
    sent = rho <= 10 ** (tau_db / 10) / g_min if np.isfinite(tau_db) else np.ones_like(rho, bool)
    power = np.where(sent, np.minimum(rho, 1 / g_min), 0)
    reached = sent & (power >= need * (1 - 1e-9))
    return dict(reliability=reached[:, servable].mean(1), power=power.mean(1) / np.where(servable, need, 0).mean(),
                withheld=1 - sent.mean(1))

def calibrate(estimate, gain, g_min: float, target: float, margins_db=MARGINS_DB, taus_db=TAUS_DB) -> dict:
    """The margin `κ` and threshold `τ` (dB) with the least power reaching `target` on these transmissions (`met`);
    if none does, the most reliable pair."""
    best = None
    for tau in taus_db:
        o = outcome(estimate, gain, g_min, tau, margins_db)
        for k, margin in enumerate(np.atleast_1d(margins_db)):
            met = bool(o['reliability'][k] >= target)
            key = (met, -o['power'][k] if met else o['reliability'][k])
            if best is None or key > best[0]:
                best = (key, dict(margin_db=float(margin), tau_db=float(tau), met=met, reliability=float(o['reliability'][k]),
                                  power=float(o['power'][k]), withheld=float(o['withheld'][k])))
    return best[1]

def evaluate(estimate, gain, g_min: float, margin_db: float, tau_db: float) -> dict:
    "Reliability, power (× perfect CSI) and withheld share at one margin and threshold, and the share of servable transmissions."
    o = outcome(estimate, gain, g_min, tau_db, [margin_db])
    return dict(reliability=float(o['reliability'][0]), power=float(o['power'][0]), withheld=float(o['withheld'][0]),
                servable=float((gain >= g_min).mean()))

### Over layouts

`calibrate` reaches the target over the calibration transmissions pooled: a new layout gets it on average, while
some layouts get less. `calibrate_layouts` asks it of each layout instead: the cheapest `κ` and `τ` with which at
most `m` of the `n` calibration layouts miss the target. If a new layout is drawn as the calibration layouts are
(exchangeable), it then reaches the target with probability at least `1 - (m + 1) / (n + 1)` (a conformal bound,
in expectation over the calibration layouts); `layouts_allowed_below` gives the `m` for a `coverage` (1 - δ), or, with
`confidence` (1 - α), the `m` for which that coverage holds with probability `1 - α` over the calibration layouts
(a binomial tolerance bound, more conservative). A layout's reliability is measured over its transmissions, so its
episodes should be many: a layout measured over a few is missed by chance as well.

In [ ]:
#| export
def layouts_allowed_below(n: int, coverage: float, confidence: float | None = None) -> int:
    """The number `m` of the `n` calibration layouts that may miss the target for a new layout to reach it with
    probability ≥ `coverage`: in expectation over the calibration layouts ((m + 1) / (n + 1) ≤ 1 - coverage), or with
    probability ≥ `confidence` over them (P(Binomial(n, 1 - coverage) ≤ m) ≤ 1 - confidence). -1: not even 0 will do."""
    if confidence is None:
        return int(math.floor((1 - coverage) * (n + 1) + 1e-9)) - 1
    from scipy.stats import binom
    m = -1
    while m + 1 < n and binom.cdf(m + 1, n, 1 - coverage) <= 1 - confidence:
        m += 1
    return m

def layout_outcome(estimate, gain, layout, g_min: float, tau_db: float = math.inf, margins_db=MARGINS_DB,
                   block: int = 20000) -> dict:
    """For each margin, as `outcome`, and per layout (`layouts`: the layouts, `per_layout`: (margins, layouts) shares
    of their servable transmissions reaching Γ, `servable`: their numbers); in blocks of transmissions."""
    margins_db = np.atleast_1d(np.asarray(margins_db, float))
    layouts, column = np.unique(layout, return_inverse=True)
    reached = np.zeros((len(margins_db), len(layouts)))
    power, sent_n = np.zeros(len(margins_db)), np.zeros(len(margins_db))
    servable_all = gain >= g_min
    for s in range(0, len(gain), block):
        e, g = np.maximum(estimate[s:s + block], 1e-30), gain[s:s + block]
        rho = 10 ** (margins_db[:, None] / 10) / e[None, :]
        sent = rho <= 10 ** (tau_db / 10) / g_min if math.isfinite(tau_db) else np.ones_like(rho, bool)
        p = np.where(sent, np.minimum(rho, 1 / g_min), 0)
        ok = (sent & (p >= (1 / g) * (1 - 1e-9))) & servable_all[s:s + block]
        onehot = np.zeros((len(g), len(layouts)), np.float32)
        onehot[np.arange(len(g)), column[s:s + block]] = 1
        reached += ok.astype(np.float32) @ onehot
        power += p.sum(1)
        sent_n += sent.sum(1)
    servable = np.bincount(column, weights=servable_all, minlength=len(layouts))
    need = np.where(servable_all, 1 / gain, 0).mean()
    return dict(reliability=reached.sum(1) / max(servable_all.sum(), 1), power=power / len(gain) / need,
                withheld=1 - sent_n / len(gain), layouts=layouts, servable=servable,
                per_layout=reached / np.maximum(servable, 1))

def calibrate_layouts(
    estimate, gain, layout, # ĝ, |h|² and layout of each calibration transmission
    g_min: float, target: float, # The weakest servable gain, and the reliability each layout should reach
    coverage: float = 0.9, # Share of new layouts that should reach it (1 - δ)
    confidence: float | None = None, # With this probability over the calibration layouts (None: in expectation)
    margins_db=MARGINS_DB, taus_db=TAUS_DB, # Tried
    min_servable: int = 20, # Layouts with fewer servable transmissions are left out
) -> dict:
    """The margin `κ` and threshold `τ` (dB) with the least power with which at most `allowed` of the calibration
    layouts miss `target` (`met`); if none, the pair with the fewest missing (then the least power)."""
    best = None
    for tau in taus_db:
        o = layout_outcome(estimate, gain, layout, g_min, tau, margins_db)
        keep = o['servable'] >= min_servable
        n = int(keep.sum())
        allowed = layouts_allowed_below(n, coverage, confidence)
        below = (o['per_layout'][:, keep] < target - 1e-12).sum(1)
        for k, margin in enumerate(np.atleast_1d(margins_db)):
            met = bool(allowed >= 0 and below[k] <= allowed)
            key = (met, -o['power'][k]) if met else (False, -below[k], -o['power'][k])
            if best is None or key > best[0]:
                best = (key, dict(margin_db=float(margin), tau_db=float(tau), met=met, below=int(below[k]),
                                  allowed=int(allowed), layouts=n, reliability=float(o['reliability'][k]),
                                  power=float(o['power'][k]), withheld=float(o['withheld'][k]),
                                  worst=float(o['per_layout'][k, keep].min()) if n else math.nan))
    return best[1]

In [ ]:
# the conformal and binomial m; layout_outcome as outcome per layout; the chosen pair the cheapest with ≤ m layouts missing
assert layouts_allowed_below(30, 0.9) == 2 and layouts_allowed_below(9, 0.9) == 0 and layouts_allowed_below(8, 0.9) == -1
assert layouts_allowed_below(30, 0.9, 0.95) == 0 and layouts_allowed_below(100, 0.9, 0.95) == 4
rng = np.random.default_rng(0)
L, per = 40, 300
layout = np.repeat(np.arange(L), per)
gain = 10 ** (rng.uniform(-28, -12, L * per) / 10)
offset = rng.normal(0, 2, L)[layout]                                   # each layout's own bias: pooled calibration misses some
estimate = gain * 10 ** ((offset + 4 * rng.standard_normal(L * per)) / 10)
g_min = 10 ** -2.6
o = layout_outcome(estimate, gain, layout, g_min, 3.0, MARGINS_DB[::8], block=777)
ref = outcome(estimate, gain, g_min, 3.0, MARGINS_DB[::8])
assert np.allclose(o['reliability'], ref['reliability']) and np.allclose(o['power'], ref['power'])
for k in (0, 7, 31):
    on = layout == k
    assert np.allclose(o['per_layout'][:, k], outcome(estimate[on], gain[on], g_min, 3.0, MARGINS_DB[::8])['reliability'])
pooled = calibrate(estimate, gain, g_min, 0.95)
per_layout = calibrate_layouts(estimate, gain, layout, g_min, 0.95, coverage=0.9)
check = layout_outcome(estimate, gain, layout, g_min, per_layout['tau_db'], [per_layout['margin_db']])['per_layout'][0]
assert per_layout['met'] and (check < 0.95).sum() == per_layout['below'] <= per_layout['allowed'] == 3
missed = layout_outcome(estimate, gain, layout, g_min, pooled['tau_db'], [pooled['margin_db']])['per_layout'][0]
assert (missed < 0.95).sum() > 3 and per_layout['power'] > pooled['power']        # pooled: more layouts miss; per layout: more power

## Saving and loading

A trained power control is the model, its history length, the chosen margin and threshold, and the budget they were
chosen for. `save_power_control` pickles them together.

In [ ]:
#| export
def save_power_control(path, model, history: int, calibration: dict, budget: Budget, **info):
    "Save the model, its history length, the calibration (`calibrate`'s) and the budget, with any other `info`."
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(dict(model=model, history=history, calibration=calibration, budget=asdict(budget), info=info), f)

def load_power_control(path) -> dict:
    "What `save_power_control` saved (the budget as a `Budget`)."
    with open(path, 'rb') as f:
        saved = pickle.load(f)
    saved['budget'] = Budget(**saved['budget'])
    return saved

## The power control, and the link as blocks

C3-WM's link (Sections III-B and IV-B) is a chain of steps, and each one is a comm-core block (see comm-core's example
*C3-WM's link, built from blocks*). Given to a `PipelineBackend`, each step can be replaced on its own:

| C3-WM | | block |
|---|---|---|
| the channel `h_t` between the agents' cells, constant during a message | eq. 4 | a `Fading` (e.g. `RayTracedFading`, or `comm.evaluation.TableFading`) |
| the power, `ρ̂ = 10^(κ/10) Γσ²/ĝ`, from a predicted gain `ĝ` | eqs. 24–26 | a `PowerControl`: `LearnedChannelInversion`, or a baseline |
| the scheduling, `δ = 1(ρ̂ ≤ 10^(τ/10) ρ_max)`, sent at no more than `ρ_max` | eq. 27 | `Schedule` |
| the SNR, `|h|² ρ / σ²` | eq. 6 | `SINR` |
| the tokens, `log₂ K` bits each, BPSK, coherent detection (optionally a CRC) | eqs. 3–5 | `UncodedChain(IndexCodec(K), BPSK())` |
| the received tokens, or the out-of-vocabulary token when nothing arrived | eq. 13 | `Tokens` |
| the communication cost | Fig. 8b | `Airtime`, and the network's metrics |

`c3wm_link` lists them. TDMA gives every directed link its own sub-slot, so there is no `Interference` block.

**`LearnedChannelInversion`** is the power: a comm-core `PowerControl`. At every round (`update`), it notes where the
nodes are and their headings, from the application (`positions()`, `headings()`); the receiver's are used a round
late, as its report would arrive. Each link's past gains are noted after each transmission, as the receiver feeds them
back. Until a link has `history` past gains, it asks for `fallback_power`. With `geometry=False`, it is the
history-only model: the past gains and the moves, one prediction (no receiver cells, no walls).

**`Schedule`** decides whether to send: only if the power asked for is within `10^(τ/10) ρ_max`, at no more than
`ρ_max`. `τ = 0` is C3-WM's `δ`; `τ = ∞` never withholds (best effort).

**`Tokens`** is what the receiver's world model gets: C3-WM has no CRC, so whatever tokens are detected, errors
included; when nothing arrives (withheld, or dropped by a CRC), the out-of-vocabulary token `OOV` in each place. It
counts the wrong tokens into the results' metadata.

In [ ]:
#| export
from comm_core import BPSK, Airtime, Block, IndexCodec, PowerControl, UncodedChain

class LearnedChannelInversion(PowerControl):
    """Eq. 26 with the gain from `model` (features -> dB): the lowest prediction over the receiver's possible cells. A
    per-step model (`steps`, `comm.sequence`) also gets the link's past steps, rebuilt from the rounds they were sent in."""

    def __init__(
        self,
        model, # Predicts the gain (dB) from `features` (or, with `geometry=False`, from the past gains and the moves)
        walls: np.ndarray, # The layout's walls: walls[x, y] = 1 where an agent cannot stand
        positions, # positions() -> {node: (x, y)}, where the nodes are now
        headings, # headings() -> {node: heading}, MultiGrid's 0-3
        history: int = 4, # Past gains the model needs
        target_snr_db: float = 10.0, # Γ
        margin_db: float = 0.0, # κ
        fallback_power: float = math.inf, # The power asked for while a link has fewer than `history` past gains (e.g. ρ_max)
        geometry: bool = True, # False: the history-only model (no receiver cells, no walls)
    ):
        self.model, self.walls, self.positions, self.headings = model, walls, positions, headings
        self.history, self.target_snr_db, self.margin_db = history, target_snr_db, margin_db
        self.fallback_power, self.geometry = fallback_power, geometry
        self.reset()

    @classmethod
    def from_saved(cls, saved: dict, walls, positions, headings):
        "From `load_power_control`'s result, for a layout's `walls` (its `τ` is `Schedule`'s: see `c3wm_link`)."
        b, c = saved['budget'], saved['calibration']
        return cls(saved['model'], walls, positions, headings, saved['history'], b.snr_target_db, c['margin_db'],
                   b.max_power_w)

    def children(self):
        return []                                                   # the model and the callables hold no blocks

    def reset(self, seed=None):
        self._past, self._rounds, self._round = {}, [], 0
        self.predicted = None

    def update(self, time):
        "A new round: where the nodes are now, and their headings (the rounds a link's history spans are kept)."
        self._round += 1
        self._rounds = (self._rounds + [({k: tuple(int(v) for v in p) for k, p in self.positions().items()},
                                          dict(self.headings()))])[-max(3, self.history + 2):]

    def _cells(self, r):
        "Where the nodes were in round `r` (None if not kept)."
        k = r - self._round + len(self._rounds) - 1
        return self._rounds[k][0] if 0 <= k < len(self._rounds) else None

    def _steps(self, past, link) -> np.ndarray:
        "The link's past steps (`step_features`, most recent first): each past gain, the agents' cells in its round, their moves."
        rows = []
        for g, r in past[-self.history:][::-1]:
            at, before = self._cells(r), self._cells(r - 1)
            moved = lambda node: before is not None and node in before and before[node] != at[node]
            rows.append(step_features(self.walls, at[link.sender], at[link.receiver], g, moved(link.sender), moved(link.receiver)))
        return np.array(rows, np.float32)

    def _rows(self, link, past) -> np.ndarray:
        "The features to each of the receiver's possible cells (`features`'; the history-only model: one row)."
        (earlier, _), (before, heads), (now, _) = self._rounds[-3:]
        gains = [g for g, _ in past[-self.history:][::-1]]
        tx, rx = now[link.sender], before[link.receiver]
        moved = (tx != before[link.sender], rx != earlier[link.receiver])
        if not self.geometry:
            return np.array([[*db(gains), float(moved[0]), float(moved[1])]])
        d = DIRS[int(heads[link.receiver])]
        free = lambda c: 0 <= c[0] < self.walls.shape[0] and 0 <= c[1] < self.walls.shape[1] and not self.walls[c]
        possible = [c for c in (rx, (rx[0] + d[0], rx[1] + d[1])) if free(c) and c != tx] or [rx]
        return np.array([features(self.walls, tx, c, gains, *moved) for c in possible])

    def allocate(self, gain, noise_power, link=None, message=None):
        past = self._past.setdefault((link.sender, link.receiver), [])
        if len(past) < self.history or len(self._rounds) < 3:
            self.predicted = None
            power = self.fallback_power
        else:
            X = self._rows(link, past)
            S = np.repeat(self._steps(past, link)[None], len(X), 0) if getattr(self.model, 'steps', False) else None
            self.predicted = 10 ** (_predict(self.model, X, S).min() / 10)
            power = 10 ** ((self.target_snr_db + self.margin_db) / 10) * noise_power / self.predicted
        past.append((float(gain), self._round))                     # fed back after the transmission, with its round
        return power, True                                          # whether to send: `Schedule`'s


class Schedule(Block):
    "Eq. 27: send only if the power asked for is within `10^(τ/10) ρ_max`, at no more than `ρ_max`."
    outputs = ('power', 'active')

    def __init__(
        self,
        max_power: float, # ρ_max
        tau_db: float = 0.0, # τ: 0 is C3-WM's δ, inf never withholds (best effort)
    ):
        self.max_power, self.tau_db = max_power, tau_db

    def call(self, power, active):
        power = np.asarray(power, float)
        return np.minimum(power, self.max_power), np.asarray(active) & (power <= self.max_power * 10 ** (self.tau_db / 10))


OOV = -1                                                            # the out-of-vocabulary token

class Tokens(Block):
    "What the receiver's world model gets: the tokens detected, or `OOV` in each place when nothing arrived."
    outputs = ('payload', 'info')

    def call(self, transmission, success, payload, info):
        tokens, extra = [], []
        for t, ok, p, i in zip(transmission, success, payload, info):
            sent = np.asarray(t.message.payload)
            got = np.asarray(p) if ok else np.full(sent.shape, OOV)
            tokens.append(got)
            extra.append({**(i or {}), 'wrong_tokens': int((got != sent).sum()) if ok else None})
        return tokens, extra


def c3wm_link(
    fading, # The channel: a comm-core `Fading` (it is told where the nodes are: `set_positions`)
    power_control, # The power: a comm-core `PowerControl` (e.g. `LearnedChannelInversion`)
    max_power: float, # ρ_max
    tau_db: float = 0.0, # τ: 0 is C3-WM's δ, inf never withholds (best effort)
    bits_per_token: int = 8, # log₂ K: a codebook of 2**bits_per_token tokens
    crc: bool = False, # Drop the messages with bit errors (they arrive as OOV), instead of delivering them
) -> list:
    "C3-WM's link as comm-core blocks, for a `PipelineBackend`: fading, power, scheduling, bits, tokens, airtime."
    return [fading, power_control, Schedule(max_power, tau_db),
            UncodedChain(IndexCodec(2 ** bits_per_token), BPSK(), drop_corrupted=crc), Tokens(), Airtime()]

In [ ]:
show_doc(Schedule)

In [ ]:
show_doc(Tokens)

In [ ]:
show_doc(c3wm_link)

## Example

The stand-in layouts of `comm.channel_data`'s example (no ray tracing): train on some layouts, calibrate on others,
judge on the rest, then run the power control through a comm-core network on one of them.

In [ ]:
import tempfile

from c3jepa_wm.comm.channel_data import ChannelTable, build_dataset, collect_trajectories, free_cells, make_env, wall_geometry
from comm_core import FullMeshTopology, IdealProtocol, Message, Network, Node, PipelineBackend, RayTracedFading

In [ ]:
env_cfg = {'_target_': 'stable_marl.envs.multigrid.FindGoalEnv', 'agents': 2, 'size': 9, 'num_obstacles': 0, 'max_steps': 60}
tmp = Path(tempfile.mkdtemp())

def stand_in_table(seed, path):
    "Instead of tracing: free-space loss at 2.4 GHz (2 m cells) and 6 dB per wall crossed."
    env = make_env(env_cfg)
    env.reset(seed=seed)
    cells, g = free_cells(env), env.unwrapped.grid
    walls = np.array([[0 if (x, y) in set(cells) else 1 for y in range(g.height)] for x in range(g.width)], np.uint8)
    H = np.zeros((len(cells), len(cells)), complex)
    for r, b in enumerate(cells):
        for s, a in enumerate(cells):
            if a != b:
                d, w = wall_geometry(walls, a, b)
                H[r, s] = 0.125 / (4 * np.pi * 2 * d) * 10 ** (-6 * w / 20)
    np.savez_compressed(path, H=H, cells=np.array(cells), walls=walls, source=json.dumps({'seed': seed}))

SEEDS = dict(train=range(0, 12), calibration=range(12, 16), test=range(16, 20))
for seed in range(20):
    stand_in_table(seed, tmp / f'layout_{seed}.npz')
collect_trajectories(env_cfg, range(20), tmp, episodes=4, max_steps=60, policy={'random': 0.5, 'epsilon_expert': 0.5})
data = {k: build_dataset(tmp, tmp, s) for k, s in SEEDS.items()}

budget = Budget(snr_target_db=10, noise_dbm=-94, max_power_dbm=10)
model = fit_predictor(data['train']['X'], data['train']['gain'], max_iter=100)
estimate = {k: predict_gain(model, d['X_possible'], d['possible_mask']) for k, d in data.items()}
chosen = calibrate(estimate['calibration'], data['calibration']['gain'], budget.g_min, target=0.95)
result = evaluate(estimate['test'], data['test']['gain'], budget.g_min, chosen['margin_db'], chosen['tau_db'])
print('chosen on the calibration layouts:', chosen)
print('on the test layouts:', result)

/home/ahmed/miniconda3/envs/rl/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


chosen on the calibration layouts: {'margin_db': 0.25, 'tau_db': 0.0, 'met': True, 'reliability': 0.953125, 'power': 3.5833220940247084, 'withheld': 0.0}
on the test layouts: {'reliability': 0.9136253041362531, 'power': 2.27737810323454, 'withheld': 0.0, 'servable': 1.0}


With so few calibration layouts, the target is missed by a few points on new layouts: the margin is chosen to just
reach it on the calibration layouts, which say little about the next one. More calibration layouts narrow the gap (in
comm-core's example *Power control on new layouts*, 6 layouts of a 9 × 9 FindGoal landed within 0.7 points of it), and
`slack` (calibrating for a slightly higher target) covers what remains.

In [ ]:
assert chosen['met'] and result['reliability'] > 0.9 and result['power'] > 1
save_power_control(tmp / 'power_control.pkl', model, 4, chosen, budget, layouts={k: list(s) for k, s in SEEDS.items()})
saved = load_power_control(tmp / 'power_control.pkl')
assert saved['calibration'] == chosen and saved['budget'] == budget

Through a network, on a test layout, every bit over BPSK (no fading here: the table is the channel), with the
threshold chosen above:

In [ ]:
table, z = ChannelTable(tmp / 'layout_16.npz'), np.load(tmp / 'trajectories_16.npz')
agents = ['agent_0', 'agent_1']
fading_map = RayTracedFading(np.sqrt(table.G).astype(complex), table.cells)
heads = {}
control = LearnedChannelInversion.from_saved(saved, table.walls, lambda: fading_map.node_positions, lambda: heads)
backend = PipelineBackend(c3wm_link(fading_map, control, budget.max_power_w, saved['calibration']['tau_db']),
                          noise_power=budget.noise_w)
print(' → '.join(type(b).__name__ for b in backend.pipeline))
net = Network([Node(a) for a in agents], FullMeshTopology(agents), IdealProtocol(), backend)
results = []
for episode, (cells, dirs) in enumerate(zip(z['cells'], z['dirs'])):
    net.reset(seed=episode)                                     # the power control forgets the last episode
    for t in range(len(cells)):
        if (cells[t, :, 0] < 0).any():                          # an agent reached the goal
            break
        backend.set_positions({a: tuple(int(v) for v in c) for a, c in zip(agents, cells[t])})
        heads.update({a: int(d) for a, d in zip(agents, dirs[t])})
        for a, b in ((0, 1), (1, 0)):
            net.send(Message(agents[a], agents[b], payload=np.arange(49) % 256))
        results += [(t, r) for r in net.step()]
learned = [r for t, r in results if t >= 4]
sent = [r for r in learned if not r.metadata.get('outage')]
print(f"{len(learned)} transmissions after each episode's first 4 steps, {len(sent)} sent; "
      f"SNR ≥ Γ: {np.mean([r.snr_db >= budget.snr_target_db - 1e-9 for r in sent]):.3f}")
assert all(r.metadata['tx_power'] <= budget.max_power_w * (1 + 1e-9) for t, r in results)
withheld = [r for r in learned if r.metadata.get('outage')]
assert all((np.asarray(r.payload) == OOV).all() for r in withheld)          # nothing arrived: OOV
assert all(r.metadata['wrong_tokens'] is None for r in withheld)
assert all(r.metadata['wrong_tokens'] == int((np.asarray(r.payload) != np.arange(49) % 256).sum()) for r in sent if r.success)

In [ ]:
# Schedule: τ = 0 withholds beyond ρ_max, a finite τ only beyond 10^(τ/10) ρ_max, τ = ∞ never; the power is capped at ρ_max
power, active = np.array([0.5, 1.5, 5.0, 50.0]), np.array([True, True, True, False])
p, a = Schedule(1.0, 0.0).call(power, active)
assert p.tolist() == [0.5, 1.0, 1.0, 1.0] and a.tolist() == [True, False, False, False]
assert Schedule(1.0, 10 * math.log10(6)).call(power, active)[1].tolist() == [True, True, True, False]
assert Schedule(1.0, math.inf).call(power, active)[1].tolist() == [True, True, True, False]
# the history-only power control: one prediction from the past gains and the moves
class Const:
    def predict(self, X): return np.full(len(X), -80.0) if X.shape[1] == 6 else np.full(len(X), np.nan)
pos, hd = {'a': (1, 1), 'b': (3, 1)}, {'a': 0, 'b': 0}
pc = LearnedChannelInversion(Const(), np.zeros((5, 5), np.uint8), lambda: pos, lambda: hd, history=4, geometry=False, fallback_power=7.0)
link_ab = __import__('comm_core').Link('a', 'b')
for _ in range(3):
    pc.update(0)
outs = [pc.allocate(1e-8, 1e-12, link=link_ab)[0] for _ in range(5)]
assert outs[:4] == [7.0] * 4 and abs(outs[4] - 10 * 1e-12 / 1e-8) < 1e-12 * 1e6                        # Γ σ² / 10^-8